# 📊 Fase 1 & 2: Data Understanding & EDA
**Geo-Price Analyzer** — Prediksi Harga Properti Jabodetabek

---
**Sumber Data:** Dataset Kaggle — Scraping dari Rumah123.com (3.500+ listing)

**Metodologi:** CRISP-DM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
plt.rcParams.update({'figure.figsize': (12,6), 'font.size': 11})

def format_rupiah(value, _=None):
    if value >= 1e9: return f'Rp {value/1e9:.1f}M'
    elif value >= 1e6: return f'Rp {value/1e6:.0f}Jt'
    else: return f'Rp {value:,.0f}'

print('✅ Library siap!')

## 1.1 Load Dataset

In [ ]:
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'raw', 'jabodetabek_house_price.csv'))
print(f'📐 Shape: {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'📋 Kolom: {list(df.columns)}')
df.head(5)

## 1.2 Info & Statistik

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 1.3 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah': missing, 'Persen (%)': missing_pct})
missing_df[missing_df['Jumlah'] > 0].sort_values('Jumlah', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
cols_with_missing = missing[missing > 0].sort_values(ascending=False)
cols_with_missing.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Missing Values per Kolom', fontweight='bold', fontsize=14)
ax.set_ylabel('Jumlah Missing')
for i, v in enumerate(cols_with_missing.values):
    ax.text(i, v+5, f'{v}', ha='center', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 1.4 Distribusi Harga

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(df['price_in_rp'].dropna(), bins=50, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribusi Harga Properti', fontweight='bold')
axes[0].set_xlabel('Harga')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))

axes[1].hist(np.log10(df['price_in_rp'].dropna()), bins=50, color='#3498db', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribusi Harga (Log₁₀)', fontweight='bold')
axes[1].set_xlabel('Log₁₀ Harga')
plt.tight_layout()
plt.show()

print(f'Min: {format_rupiah(df["price_in_rp"].min())} | Max: {format_rupiah(df["price_in_rp"].max())}')
print(f'Mean: {format_rupiah(df["price_in_rp"].mean())} | Median: {format_rupiah(df["price_in_rp"].median())}')

## 1.5 Harga per Kota

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
order = df.groupby('city')['price_in_rp'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='city', y='price_in_rp', order=order, palette='coolwarm', ax=ax, fliersize=2)
ax.set_title('Distribusi Harga per Kota', fontweight='bold', fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Tabel median harga per kota
summary = df.groupby('city')['price_in_rp'].agg(['median','mean','count']).sort_values('median', ascending=False)
summary['median'] = summary['median'].apply(format_rupiah)
summary['mean'] = summary['mean'].apply(format_rupiah)
summary.columns = ['Median', 'Rata-rata', 'Jumlah Listing']
summary

## 1.6 Jumlah Listing per Kota

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
counts = df['city'].value_counts().sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(counts)))
counts.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Jumlah Listing per Kota', fontweight='bold', fontsize=14)
for i, v in enumerate(counts.values):
    ax.text(v+5, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 1.7 Scatter Plot: Fitur vs Harga

In [ ]:
features = ['land_size_m2', 'building_size_m2', 'bedrooms', 'bathrooms']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.ravel(), features):
    ax.scatter(df[feat], df['price_in_rp'], alpha=0.3, s=10, c='#e74c3c')
    ax.set_title(f'{feat} vs Harga', fontweight='bold')
    ax.set_xlabel(feat)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
plt.suptitle('Hubungan Fitur vs Harga', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 1.8 Correlation Heatmap

In [ ]:
num_cols = ['price_in_rp','land_size_m2','building_size_m2','bedrooms','bathrooms','carports','garages','floors']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True, ax=ax)
ax.set_title('Correlation Heatmap', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print('\n📊 Korelasi terhadap Harga:')
for feat, val in corr['price_in_rp'].drop('price_in_rp').sort_values(ascending=False).items():
    print(f'   {feat:20s}: {val:+.3f}')

## 1.9 Deteksi Outliers

In [ ]:
Q1 = df['price_in_rp'].quantile(0.25)
Q3 = df['price_in_rp'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['price_in_rp'] < lower) | (df['price_in_rp'] > upper)]

print(f'Q1 = {format_rupiah(Q1)} | Q3 = {format_rupiah(Q3)} | IQR = {format_rupiah(IQR)}')
print(f'Upper Bound = {format_rupiah(upper)}')
print(f'Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(df['price_in_rp'].dropna())
axes[0].set_title('Sebelum Remove Outliers', fontweight='bold')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
clean = df[(df['price_in_rp'] >= lower) & (df['price_in_rp'] <= upper)]
axes[1].boxplot(clean['price_in_rp'])
axes[1].set_title('Sesudah Remove Outliers', fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
plt.tight_layout()
plt.show()

---
## 📝 Kesimpulan EDA
| Aspek | Temuan |
|---|---|
| Jumlah Data | 3.500+ listing dari Rumah123.com |
| Missing Values | Ditemukan di beberapa kolom |
| Outliers | Ada harga anomali yang perlu dihapus |
| Korelasi | `land_size_m2` dan `building_size_m2` berkorelasi kuat dengan harga |

**Selanjutnya →** `02_Data_Preparation.ipynb`